# Phase 1 & 2 : Chargement, Nettoyage des Données et Analyse Exploratoire (EDA)

Ce notebook présente la première phase du projet de prévision de séries temporelles du tourisme au Maroc et l'ingénierie des variables du secteur hôtelier.

## Objectifs :
1. **Chargement et fusion** des données macroéconomiques marocaines et des arrivées touristiques.
2. **Nettoyage et Imputation** : Intégration des données COVID réelles, gestion des valeurs manquantes.
3. **Reconstruction historique** : Simulation mensuelle des arrivées et recettes (1996-2019) à partir des totaux annuels et des profils saisonniers récents.
4. **Analyse Exploratoire (EDA)** : Décomposition saisonnière (STL) et analyses de stationnarité (ADF, KPSS, PP).
5. **Analyse Hôtelière** : Agrégation des données hôtelières, calcul du profil saisonnier et benchmark comparatif avec d'autres destinations compétitives (Égypte, Turquie, Espagne, France, Grèce, EAU).

In [ ]:
# Activation de l'autoreload pour recharger automatiquement les modules src/ modifiés
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from scipy.stats import shapiro, linregress

# Importation de nos modules personnalisés propres
from src.config import DATA_DIR, TARGET_COL
import src.data_loader as loader
import src.cleaning as cleaner
import src.visualization as viz

## 1. Chargement et Fusion des Données Touristiques et Économiques

Nous lisons et fusionnons les données macroéconomiques de base (`Morocco_cleaned.csv`) et le jeu de données d'arrivées touristiques (`maroc_tourism_2030_all_arrival_sources.csv`). Le code gère automatiquement le nettoyage des colonnes de recettes dupliquées et s'adapte à l'environnement d'exécution (Colab ou local).

In [ ]:
merged_df = loader.load_and_merge_tourism_data()
print(f"Dimensions du dataset fusionné initial : {merged_df.shape}")
merged_df.head()

## 2. Intégration des Données COVID Réelles

Nous écrasons les valeurs aberrantes ou manquantes des années 2020-2021 avec les données COVID réelles collectées manuellement, et nous activons le flag d'événement `is_covid` pour marquer ce choc exogène.

In [ ]:
merged_df = cleaner.integrate_covid_data(merged_df)
print("Aperçu des données pendant la période COVID (2020-2021) :")
merged_df[(merged_df['Date'] >= '2020-03-01') & (merged_df['Date'] <= '2021-12-01')].head()

## 3. Reconstruction Historique (Simulations 1996-2019)

Comme les arrivées et recettes mensuelles historiques sont manquantes (seuls les totaux annuels sont fiables avant 2020), nous utilisons le profil saisonnier extrait de la période récente (2022-2026) par décomposition multiplicative pour désagréger les totaux annuels historiques en données mensuelles réalistes avec un bruit gaussien résiduel.

In [ ]:
# A. Reconstruction des Arrivées
merged_df = cleaner.reconstruct_historical_arrivals(merged_df)

# B. Reconstruction des Recettes Touristiques Totales
merged_df = cleaner.reconstruct_historical_receipts(merged_df)

print(f"Valeurs manquantes restantes dans Arrivals : {merged_df['Arrivals'].isna().sum()}")
print(f"Valeurs manquantes restantes dans Total_Receipts_MDH : {merged_df['Total_Receipts_MDH'].isna().sum()}")

## 4. Sauvegarde des Données Touristiques Nettoyées

Sauvegarde du jeu de données principal fusionné dans le dossier de données.

In [ ]:
output_file = os.path.join(DATA_DIR, 'merged_tourism_data_final.csv')
merged_df.to_csv(output_file, index=False)
print(f"Fichier sauvegardé avec succès -> {output_file}")
merged_df.tail()

## 5. Analyse Exploratoire (EDA) & Visualisations

Tracé de l'évolution des arrivées et application d'une décomposition saisonnière (STL) additive pour analyser la tendance à long terme, la saisonnalité annuelle stable et les résidus (bruit).

In [ ]:
# Tracé des arrivées historiques et récentes avec le choc COVID en rouge
viz.plot_arrivals_evolution(merged_df, title="Évolution des Arrivées Touristiques au Maroc (1995-2026)")

In [ ]:
# Décomposition saisonnière de la série des arrivées
decomp_data = merged_df.set_index('Date')['Arrivals']
decomposition = seasonal_decompose(decomp_data, model='additive', period=12)
viz.plot_seasonal_decomposition(decomposition, title="Décomposition Additive des Arrivées Touristiques")

In [ ]:
# Corrélation entre variables macroéconomiques et touristiques
corr_columns = ['REER', 'Oil_price', 'FDI', 'Poverty_rate', 'Arrivals', 'Total_Receipts_MDH', 'is_covid']
viz.plot_correlation_matrix(merged_df, corr_columns, title="Matrice de Corrélation des Variables Économiques")

### Tests de Stationnarité sur les Arrivées

Nous effectuons les tests de Dickey-Fuller Augmenté (ADF) et KPSS sur la série brute des arrivées pour évaluer la nécessité d'une différenciation.

In [ ]:
arrivals_series = merged_df['Arrivals'].dropna()

print("--- Test de Dickey-Fuller Augmenté (ADF) ---")
adf_stat, adf_p, *rest = adfuller(arrivals_series)
print(f"Statistique ADF : {adf_stat:.4f} | p-value : {adf_p:.4e}")
print("Stationnaire (p < 0.05)" if adf_p < 0.05 else "Non stationnaire (p >= 0.05)")

print("\n--- Test KPSS ---")
kpss_stat, kpss_p, *rest = kpss(arrivals_series, regression='c', nlags='auto')
print(f"Statistique KPSS : {kpss_stat:.4f} | p-value : {kpss_p:.4f}")
print("Stationnaire (p > 0.05)" if kpss_p > 0.05 else "Non stationnaire (p <= 0.05) - Présence d'une racine unité")

## 6. Analyse Hôtelière & Profils Saisonniers

Nous analysons le comportement de l'hôtellerie locale à partir de `hotel_bookings_clean.csv`. Nous rééchantillonnons de manière appropriée en utilisant la moyenne pour les taux (`occupancy_rate`, `cancel_rate`) et la somme pour les volumes (`total_revenue`).

In [ ]:
raw_hotel = loader.load_hotel_bookings()
hotel_monthly = cleaner.clean_and_resample_hotel_data(raw_hotel)
print(f"Dimensions des données hôtelières mensuelles : {hotel_monthly.shape}")
hotel_monthly.head()

In [ ]:
# Visualisation des tendances mensuelles hôtelières
viz.plot_hotel_trends(hotel_monthly)

In [ ]:
# Calcul du profil saisonnier de l'hôtellerie locale
seasonal_profile = cleaner.get_hotel_seasonal_profile(hotel_monthly)
print("Profil saisonnier hôtelier calculé :")
display(seasonal_profile)

# Sauvegarde du profil saisonnier
seasonal_profile.to_csv('hotel_seasonal_profile.csv', index=False)
print("Profil saisonnier hôtelier sauvegardé dans hotel_seasonal_profile.csv")

## 7. Analyses de Benchmark de l'Hospitalité

Nous comparons la performance de récupération de l'hôtellerie du Maroc avec d'autres destinations phares de la région méditerranéenne et du Moyen-Orient.

In [ ]:
raw_bench = loader.load_hospitality_benchmark()
bench_comp, bench_monthly = cleaner.process_hospitality_benchmark(raw_bench)

print("Ratio de récupération moyen en 2022 par pays (par rapport à la baseline pré-COVID) :")
recovery_2022 = bench_comp[bench_comp['date'].dt.year == 2022].groupby('Country')['occ_recovery_ratio'].mean().sort_values(ascending=False)
print(recovery_2022.round(3).to_string())

# Tracé comparatif
comparable_countries = ['Egypt', 'Turkey', 'Spain', 'France', 'UAE', 'Greece']
viz.plot_benchmark_occupancy(bench_comp, comparable_countries)

In [ ]:
# Corrélation de l'occupation entre pays compétiteurs
bench_pivot = bench_comp.pivot_table(index='date', columns='Country', values='occupancy', aggfunc='mean')
viz.plot_correlation_matrix(bench_pivot, comparable_countries, title="Corrélations d'Occupation entre Pays")

In [ ]:
# Régression de la satisfaction sur le taux d'occupation
valid_data = bench_comp.dropna(subset=['occupancy', 'satisfaction'])
slope, intercept, r, p, se = linregress(valid_data['occupancy'], valid_data['satisfaction'])
print(f"Régression Satisfaction ~ Occupancy :\nSlope = {slope:.4f} | Coeff de Corrélation R = {r:.3f} | p-value = {p:.4e}")

# Sauvegarde du benchmark comparable pour imputation future
bench_monthly.to_csv('benchmark_comparable.csv', index=False)
print("Benchmark comparable mensuel exporté avec succès dans benchmark_comparable.csv")